In [ ]:
!pip install typhoon-ocr

In [ ]:
!apt-get update -qq && apt-get install -qq poppler-utils

In [ ]:
import os
api_key = os.getenv('TYPHOON_OCR_API_KEY')

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
filename = next(iter(uploaded))

In [ ]:
from typhoon_ocr import ocr_document

In [ ]:
markdown = ocr_document(
    pdf_or_image_path=filename,  # ใช้ได้กับ PDF หรือรูปภาพ
    task_type="default",    # เลือกระหว่าง "default" หรือ "structure"
)

In [ ]:
print("----กรุณาตรวจสอบก่อนนำข้อมูลไปใช้----")
print(markdown)

In [ ]:
#Another case ถ้าต้องการส่งเป็น JSON (กรุณาตรวจสอบก่อนนำข้อมูลไปใช้)
import re
import json

def extract_police_report_v2(text):
    # ล้างข้อมูล: ลบช่องว่างซ้ำๆ และ Newline
    text = re.sub(r'\s+', ' ', text).strip()

    data = {
        "reporter": {
            "name": None, "age": None, "id_card": None, "address": None, "phone": None
        },
        "missing_person": {
            "name": None, "age": None, "details": None
        }
    }

    # --- 1. แบ่งโซนข้อความ ---
    # เพิ่มคำว่า "แจ้งว่า" เผื่อ OCR ตัดคำผิด
    split_point = re.search(r"(?:ได้?มา?พบ?พนักงานสอบสวน|แจ้งว่า)", text)

    if split_point:
        reporter_text = text[:split_point.start()]
        missing_text = text[split_point.end():]
    else:
        reporter_text = text
        missing_text = ""

    # --- 2. ดึงข้อมูลผู้แจ้ง ---

    # ชื่อผู้แจ้ง (ให้รองรับการขึ้นต้นด้วย นาย/นาง เลย โดยไม่ต้องมีคำว่า ข้าพเจ้า)
    match_r_name = re.search(r"(?:ข้าพเจ้า|ผู้แจ้ง|ชื่อ|^)\s*(?:นาย|นาง|นางสาว|น\.ส\.|ยศ\.|ด\.ต\.|ร\.ต\.อ\.)\s*([^\s0-9]+(?:\s+[^\s0-9]+)?)", reporter_text)
    if match_r_name:
        data["reporter"]["name"] = match_r_name.group(1).strip()

    # อายุ
    match_r_age = re.search(r"อายุ\s*(\d{1,3})\s*ปี", reporter_text)
    if match_r_age:
        data["reporter"]["age"] = match_r_age.group(1)

    # เลขบัตร
    match_r_id = re.search(r"(\d{1}\s?-?\s?\d{4}\s?-?\s?\d{5}\s?-?\s?\d{2}\s?-?\s?\d{1})", reporter_text)
    if match_r_id:
        data["reporter"]["id_card"] = re.sub(r"[-\s]", "", match_r_id.group(1))

    # ที่อยู่
    match_r_addr = re.search(r"(?:ที่อยู่|อยู่บ้านเลขที่|พักอาศัย|อยู่บ้าน)\s*(.+?)(?=\s+(?:เบอร์|โทรศัพท์|เกี่ยวข้อง|ได้มา|มาพบ))", reporter_text)
    if match_r_addr:
        data["reporter"]["address"] = match_r_addr.group(1).strip()

    # เบอร์โทร
    match_r_phone = re.search(r"(?:โทรศัพท์|เบอร์|โทร\.?|มือถือ)\s*[:.]?\s*(\d{2,3}[\s-]?\d{3}[\s-]?\d{4})", reporter_text)
    if match_r_phone:
        data["reporter"]["phone"] = re.sub(r"[-\s]", "", match_r_phone.group(1))

    # --- 3. ดึงข้อมูลคนหาย ---

    # ชื่อคนหาย (เพิ่ม บ.ส. เผื่อ OCR เพี้ยน และปรับให้หาคำนำหน้าได้กว้างขึ้น)
    match_m_name = re.search(r"(?:เด็กชาย|เด็กหญิง|ด\.ช\.|ด\.ญ\.|นาย|นาง|นางสาว|น\.ส\.|บ\.ส\.|บุตร|หลาน)\s*([^\s0-9]+(?:\s+[^\s0-9]+)?)", missing_text)
    if match_m_name:
        data["missing_person"]["name"] = match_m_name.group(1).strip()

    # อายุคนหาย
    match_m_age = re.search(r"อายุ\s*(\d{1,3})\s*(?:ปี|ขวบ|เดือน)", missing_text)
    if match_m_age:
        data["missing_person"]["age"] = match_m_age.group(1)

    # รายละเอียด
    if match_m_name:
         start_detail = missing_text.find(match_m_name.group(1)) + len(match_m_name.group(1))
         data["missing_person"]["details"] = missing_text[start_detail:].strip()

    return data

if 'markdown' in globals():
    result = extract_police_report_v2(markdown)
    print("--- ข้อมูลที่ได้ ---")
    print(json.dumps(result, indent=4, ensure_ascii=False))
else:
    print("ไม่พบตัวแปร markdown กรุณารัน cell OCR ด้านบนก่อน")